# Data Source

This notebook prepares the raw ITBI transaction data for the next stages of the project. 
The goal is to understand the dataset, fix formatting issues, identify duplicated information, and save a cleaner version for analysis and modeling.

The data come from the [Rio de Janeiro City Hall ITBI dataset](https://fazenda.prefeitura.rio/itbi-valores-medios-por-m2-de-transacoes-imobiliarias-por-trecho-de-logradouro/).

The sourc ein slipt in two Excel files: 
- from 2016 to 2024: *ITBI_TRANSACOES_2016_2024_ver20260807094924-1.xlsx*
- from 2025: *ITBI_TRANSACOES_2025_diante-5.xlsx*

(ITBI – Valores médios por m² de transações imobiliárias por trecho de logradouro)

In [3]:
import pandas as pd

# Importing the data

The two source files cover consecutive periods, so they are loaded and concatenated into a single DataFrame.

In [9]:
# filepaths of both ITBI files
rawdata1_filepath = "../data/raw/ITBI_TRANSACOES_2016_2024_ver20260807094924-1.xlsx"
rawdata2_filepath = "../data/raw/ITBI_TRANSACOES_2025_diante-5.xlsx"

# import spreadsheets as DataFrames
df_raw = pd.concat(
    [
        pd.read_excel(rawdata1_filepath), # from 2016 to 2024
        pd.read_excel(rawdata2_filepath)  # from 2025
    ]
)
df_raw.info()

<class 'pandas.DataFrame'>
Index: 490371 entries, 0 to 79934
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ano                  490371 non-null  int64  
 1   mes                  490371 non-null  int64  
 2   ano_texto            490371 non-null  str    
 3   mes_texto            490371 non-null  str    
 4   COD_REGIAO_FISCAL    490371 non-null  str    
 5   COD_BAIRRO           490274 non-null  str    
 6   COD_LOGRADOURO       490323 non-null  str    
 7   COD_TRECHO           481702 non-null  str    
 8   COD_TIPOLOGIA        490371 non-null  str    
 9   ID_UTILIZACAO        490371 non-null  int64  
 10  NUMERO               490371 non-null  int64  
 11  COMPLEMENTO          470044 non-null  str    
 12  FATOR_IDADE          477590 non-null  float64
 13  nm_bairro_norm       490199 non-null  str    
 14  nm_logradouro        490077 non-null  str    
 15  ds_imovel_tipologia  490371 non-nu

In [48]:
df_raw.head(5)

,ano,mes,ano_texto,mes_texto,COD_REGIAO_FISCAL,COD_BAIRRO,COD_LOGRADOURO,COD_TRECHO,COD_TIPOLOGIA,ID_UTILIZACAO,...,COMPLEMENTO,FATOR_IDADE,nm_bairro_norm,nm_logradouro,ds_imovel_tipologia,DESCRICAO,PARTE_TRANSFERIDA,vl_declarado,area,valor_m2
0,2016,1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_58396,_x0030_01,_x0031_0,1,...,APT_x0020_204_x0020_BLC_x0020_1,0.80,GUADALUPE,Rua_x0020_Leocádio_x0020_Figueiredo,APARTAMENTO,RESIDENCIAL,_x0031_00,70150.00,51.0,1375.490196
1,2016,1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_92833,_x0030_01,_x0038_1,1,...,NaN,0.51,GUADALUPE,Rua_x0020_Enéas_x0020_Martins,PROLETARIO_x0020_CASA_x002F_APTO,RESIDENCIAL,_x0031_00,28158.86,36.0,782.190556
2,2016,1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93435,_x0030_01,_x0031_0,1,...,BLOCO_x0020_4_x0020_APTO_x0020_102,1.00,GUADALUPE,Rua_x0020_João_x0020_Paranaguá,APARTAMENTO,RESIDENCIAL,_x0031_00,186716.00,48.0,3889.916667
3,2016,1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93435,_x0030_01,_x0031_0,1,...,BLOCO_x0020_5_x0020_APTO_x0020_302,1.00,GUADALUPE,Rua_x0020_João_x0020_Paranaguá,APARTAMENTO,RESIDENCIAL,_x0031_00,188100.00,48.0,3918.750000
4,2016,1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93484,_x0030_01,_x0038_1,1,...,NaN,0.65,GUADALUPE,Rua_x0020_Joaquim_x0020_Sarmento,PROLETARIO_x0020_CASA_x002F_APTO,RESIDENCIAL,_x0031_00,50000.00,64.0,781.250000


# Fixing character-encoding issues

Some columns that should contain numerical codes were imported as strings. 
Several of these strings also contain Excel character-encoding artifacts.

For example, `_x0031_` represents the digit 1. 
Therefore, a value such as `_x0031_06` should be read as 106.

Before changing data types, let's inspect the columns stored as strings and check how widespread the issue is.

In [51]:
str_raw_features = ['ano_texto', 'mes_texto', 'COD_REGIAO_FISCAL',
       'COD_BAIRRO', 'COD_LOGRADOURO', 'COD_TRECHO', 'COD_TIPOLOGIA', 'COMPLEMENTO',
        'nm_bairro_norm', 'nm_logradouro', 'ds_imovel_tipologia', 'DESCRICAO',
       'PARTE_TRANSFERIDA']
df_raw[str_raw_features].head(15)

,ano_texto,mes_texto,COD_REGIAO_FISCAL,COD_BAIRRO,COD_LOGRADOURO,COD_TRECHO,COD_TIPOLOGIA,COMPLEMENTO,nm_bairro_norm,nm_logradouro,ds_imovel_tipologia,DESCRICAO,PARTE_TRANSFERIDA
0,_x0032_016,_x0030_1,A,_x0031_06,_x0030_58396,_x0030_01,_x0031_0,APT_x0020_204_x0020_BLC_x0020_1,GUADALUPE,Rua_x0020_Leocádio_x0020_Figueiredo,APARTAMENTO,RESIDENCIAL,_x0031_00
1,_x0032_016,_x0030_1,A,_x0031_06,_x0030_92833,_x0030_01,_x0038_1,NaN,GUADALUPE,Rua_x0020_Enéas_x0020_Martins,PROLETARIO_x0020_CASA_x002F_APTO,RESIDENCIAL,_x0031_00
2,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93435,_x0030_01,_x0031_0,BLOCO_x0020_4_x0020_APTO_x0020_102,GUADALUPE,Rua_x0020_João_x0020_Paranaguá,APARTAMENTO,RESIDENCIAL,_x0031_00
3,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93435,_x0030_01,_x0031_0,BLOCO_x0020_5_x0020_APTO_x0020_302,GUADALUPE,Rua_x0020_João_x0020_Paranaguá,APARTAMENTO,RESIDENCIAL,_x0031_00
4,_x0032_016,_x0030_1,A,_x0031_06,_x0030_93484,_x0030_01,_x0038_1,NaN,GUADALUPE,Rua_x0020_Joaquim_x0020_Sarmento,PROLETARIO_x0020_CASA_x002F_APTO,RESIDENCIAL,_x0031_00
5,_x0032_016,_x0030_1,A,_x0031_06,_x0030_96925,_x0030_02,_x0030_2,NaN,GUADALUPE,Rua_x0020_Francisco_x0020_Portela,CASA,RESIDENCIAL,_x0031_00
6,_x0032_016,_x0030_1,A,_x0031_07,_x0030_33977,_x0030_01,_x0033_6,LOJ_x0020_A,ANCHIETA,Rua_x0020_Motor_x0020_Luis_x0020_Abreu,LOJA,NAO_x0020_RESIDENCIAL,_x0031_00
7,_x0032_016,_x0030_1,A,_x0031_07,_x0030_36988,_x0030_01,_x0031_0,BL_x0020_3_x0020_APT_x0020_201,ANCHIETA,Avenida_x0020_Chrisóstomo_x0020_Pimentel_x0020...,APARTAMENTO,RESIDENCIAL,_x0031_00
8,_x0032_016,_x0030_1,A,_x0031_07,_x0030_37184,_x0030_01,_x0031_0,APT_x0020_201_x0020_FDS,ANCHIETA,Rua_x0020_Ciere,APARTAMENTO,RESIDENCIAL,_x0031_00
9,_x0032_016,_x0030_1,A,_x0031_07,_x0031_63683,_x0030_01,_x0030_2,NaN,ANCHIETA,Rua_x0020_Frederico_x0020_Edelweiss,CASA,RESIDENCIAL,_x0031_00


The encoding issue appears in several string columns. 
We can fix it by replacing the encoded digit sequences with their corresponding numbers.

In [58]:
def decode_excel_string(col):
    return (
        col.astype("string")
        .str.replace("_x0030_", "0")
        .str.replace("_x0031_", "1")
        .str.replace("_x0032_", "2")
        .str.replace("_x0033_", "3")
        .str.replace("_x0034_", "4")
        .str.replace("_x0035_", "5")
        .str.replace("_x0036_", "6")
        .str.replace("_x0037_", "7")
        .str.replace("_x0038_", "8")
        .str.replace("_x0039_", "9")
    )

In [67]:
for _col in str_raw_features:
    df_raw[_col] = df_raw[_col].apply(decode_excel_string)

In [70]:
df_raw[str_raw_features].head(3)

,ano_texto,mes_texto,COD_REGIAO_FISCAL,COD_BAIRRO,COD_LOGRADOURO,COD_TRECHO,COD_TIPOLOGIA,COMPLEMENTO,nm_bairro_norm,nm_logradouro,ds_imovel_tipologia,DESCRICAO,PARTE_TRANSFERIDA
0,2016,01,A,106,058396,001,10,APT 204 BLC 1,GUADALUPE,Rua Leocádio Figueiredo,APARTAMENTO,RESIDENCIAL,100
1,2016,01,A,106,092833,001,81,NaN,GUADALUPE,Rua Enéas Martins,PROLETARIO CASA/APTO,RESIDENCIAL,100
2,2016,01,A,106,093435,001,10,BLOCO 4 APTO 102,GUADALUPE,Rua João Paranaguá,APARTAMENTO,RESIDENCIAL,100


The values now look consistent. 

Next, the columns that represent numerical codes are converted from strings to numeric types. 
This also helps confirm that the encoding artifacts were removed successfully.

In [82]:
str2num_features = ['ano_texto', 'mes_texto', 'COD_BAIRRO', 'COD_LOGRADOURO', 'COD_TRECHO', 'COD_TIPOLOGIA']
df_raw[str2num_features] = df_raw[str2num_features].astype("float", errors="ignore")

In [83]:
df_raw.info()

<class 'pandas.DataFrame'>
Index: 490371 entries, 0 to 79934
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ano                  490371 non-null  int64  
 1   mes                  490371 non-null  int64  
 2   ano_texto            490371 non-null  float64
 3   mes_texto            490371 non-null  float64
 4   COD_REGIAO_FISCAL    490371 non-null  str    
 5   COD_BAIRRO           490274 non-null  float64
 6   COD_LOGRADOURO       490323 non-null  float64
 7   COD_TRECHO           481702 non-null  float64
 8   COD_TIPOLOGIA        490371 non-null  float64
 9   ID_UTILIZACAO        490371 non-null  int64  
 10  NUMERO               490371 non-null  int64  
 11  COMPLEMENTO          470044 non-null  str    
 12  FATOR_IDADE          477590 non-null  float64
 13  nm_bairro_norm       490199 non-null  str    
 14  nm_logradouro        490077 non-null  str    
 15  ds_imovel_tipologia  490371 non-nu

# Understanding the data

Before cleaning the dataset further, it is useful to understand what each feature represents:
- `ano`: year.
- `mes`: month.
- `ano_texto`: year as string type.
- `mes_texto`: month as string type.
- `COD_REGIAO_FISCAL`: fiscal code.
- `COD_BAIRRO`: neighborhood code.
- `COD_LOGRADOURO`: street identifier code.
- `COD_TRECHO`: identifier of street segment.
- `COD_TIPOLOGIA`: Property/building typology code.
- `ID_UTILIZACAO`: Numerical identification of property usage type.
- `NUMERO`: street/door number.
- `COMPLEMENTO`: address complement.
- `FATOR_IDADE`: age coefficient used in municipal property valuation.
- `nm_bairro_norm`: neighborhood name, linked to `COD_BAIRRO`.
- `nm_logradouro`: street name, linked to `COD_LOGRADOURO`.
- `ds_imovel_tipologia`: description of property/building typology.
- `DESCRICAO`: description of property usage type as stated in `ID_UTILIZACAO`.
- `PARTE_TRANSFERIDA`: fraction of the property transferred.
- `vl_declarado`: declared value for the transfaction.
- `area`: property area used in the published calculation.
- `valor_m2`: value per square meter, derived from `vl_declarado` and `area`.

## Missing values

Let's first check the number and percentage of missing values in each feature.

In [22]:
pd.concat(
    [
        df_raw.isnull().sum().sort_values(ascending=False),
        df_raw.isnull().mean().sort_values(ascending=False)*100
    ], 
    axis=1,
    keys=["counts", "%"]
)

,counts,%
COMPLEMENTO,20327,4.145229
FATOR_IDADE,12781,2.606394
COD_TRECHO,8669,1.767845
nm_logradouro,294,0.059955
nm_bairro_norm,172,0.035075
COD_BAIRRO,97,0.019781
COD_LOGRADOURO,48,0.009789
valor_m2,40,0.008157
area,40,0.008157
vl_declarado,0,0.000000


Missing values are limited overall. 
`COMPLEMENTO` has the largest number of null values, but they still represent less than 5% of the dataset.
The next step is to inspect the features with missing values and check whether the missingness follows a clear pattern.

In [142]:
df_raw[df_raw["FATOR_IDADE"].isna()].head(15)

,ano,mes,ano_texto,mes_texto,COD_REGIAO_FISCAL,COD_BAIRRO,COD_LOGRADOURO,COD_TRECHO,COD_TIPOLOGIA,ID_UTILIZACAO,...,COMPLEMENTO,FATOR_IDADE,nm_bairro_norm,nm_logradouro,ds_imovel_tipologia,DESCRICAO,PARTE_TRANSFERIDA,vl_declarado,area,valor_m2
50,2016,1,2016.0,1.0,A,141.0,148569.0,1.0,1.0,3,...,LOT 4 LTM 28955 QDR L,NaN,BANGU,Rua Dilson Avila Tome,TERRENO,TERRITORIAL,100,50000.0,120.0,416.666667
62,2016,1,2016.0,1.0,A,143.0,197228.0,1.0,1.0,3,...,LOT 41 LTM 39577 QDR D,NaN,SANTISSIMO,Rua Lidja Zamenhof,TERRENO,TERRITORIAL,100,25000.0,153.0,163.398693
63,2016,1,2016.0,1.0,A,143.0,211912.0,1.0,1.0,3,...,LOT 05 PAL 44409 QDR F,NaN,SANTISSIMO,Rua Edna Savaget,TERRENO,TERRITORIAL,100,78110.9,127.0,615.046457
111,2016,1,2016.0,1.0,A,144.0,40873.0,4.0,1.0,3,...,LOT 12 LTM 36056 QDR A,NaN,CAMPO GRANDE,Estrada do Tingui,TERRENO,TERRITORIAL,100,165000.0,360.0,458.333333
112,2016,1,2016.0,1.0,A,144.0,40873.0,4.0,1.0,3,...,LOTE 07 QD IV PAL 47027,NaN,CAMPO GRANDE,Estrada do Tingui,TERRENO,TERRITORIAL,100,18000.0,191.0,94.240838
125,2016,1,2016.0,1.0,A,144.0,130229.0,1.0,1.0,3,...,LOT 8 LTM 19789 QDR 16,NaN,CAMPO GRANDE,Rua B - PAL 19789,TERRENO,TERRITORIAL,100,82000.0,360.0,227.777778
126,2016,1,2016.0,1.0,A,144.0,130237.0,1.0,1.0,3,...,LOT 18 LTM 19789 QDR 5,NaN,CAMPO GRANDE,Rua C - PAL 19789,TERRENO,TERRITORIAL,100,55000.0,360.0,152.777778
135,2016,1,2016.0,1.0,A,144.0,173476.0,1.0,1.0,3,...,LOT 11 LTM 32309 QDR 4,NaN,CAMPO GRANDE,Rua Tomás Soares da Silva,TERRENO,TERRITORIAL,100,20000.0,120.0,166.666667
136,2016,1,2016.0,1.0,A,144.0,173534.0,1.0,1.0,3,...,LOT 2 LTM 32309 QDR 1,NaN,CAMPO GRANDE,Rua Rodrigo Ranieri,TERRENO,TERRITORIAL,100,10000.0,120.0,83.333333
137,2016,1,2016.0,1.0,A,144.0,175216.0,1.0,1.0,3,...,LOT 5 PAL 47729,NaN,CAMPO GRANDE,Rua Artur Santos,TERRENO,TERRITORIAL,100,56050.0,180.0,311.388889


## Duplicated info

Several features contain the same information in different forms. 
A common pattern is a numerical code paired with a text description.

Before removing any duplicated representation, we need to verify how these columns are related.

### Location info

The location information follows a hierarchy from broad to detailed:
- **Neighborhood** as `COD_BAIRRO` with name `nm_bairro_norm`.
    - **Street** as `COD_LOGRADOURO` with name `nm_logradouro`.
        - **Street segment** as `COD_TRECHO`.
            - **Street number** as `NUMERO`
            
The next checks verify whether the code and name columns have a consistent mapping.

In [88]:
location_features = ["COD_BAIRRO", "nm_bairro_norm", "COD_LOGRADOURO", "nm_logradouro", "NUMERO"]
df_raw[location_features].dropna().head(15)

,COD_BAIRRO,nm_bairro_norm,COD_LOGRADOURO,nm_logradouro,NUMERO
0,106.0,GUADALUPE,58396.0,Rua Leocádio Figueiredo,650
1,106.0,GUADALUPE,92833.0,Rua Enéas Martins,79
2,106.0,GUADALUPE,93435.0,Rua João Paranaguá,230
3,106.0,GUADALUPE,93435.0,Rua João Paranaguá,230
4,106.0,GUADALUPE,93484.0,Rua Joaquim Sarmento,140
5,106.0,GUADALUPE,96925.0,Rua Francisco Portela,445
6,107.0,ANCHIETA,33977.0,Rua Motor Luis Abreu,376
7,107.0,ANCHIETA,36988.0,Avenida Chrisóstomo Pimentel de Oliveira,477
8,107.0,ANCHIETA,37184.0,Rua Ciere,55
9,107.0,ANCHIETA,163683.0,Rua Frederico Edelweiss,55


Now, we check whether each neighborhood code maps to a single neighborhood name, and whether each street code maps to a single street name.

In [96]:
print(
    "Number of nm_bairro_norm categories in COD_BAIRRO:",
    df_raw.groupby("COD_BAIRRO")["nm_bairro_norm"].nunique().max()
)

print(
    "Number of COD_BAIRRO in nm_bairro_norm categories:",
    df_raw.groupby("nm_bairro_norm")["COD_BAIRRO"].nunique().max()
)

Number of nm_bairro_norm categories in COD_BAIRRO: 1
Number of COD_BAIRRO in nm_bairro_norm categories: 1


In [97]:
print(
    "Number of nm_logradouro categories in COD_LOGRADOURO:",
    df_raw.groupby("COD_LOGRADOURO")["nm_logradouro"].nunique().max()
)

print(
    "Number of COD_LOGRADOURO in nm_logradouro categories:",
    df_raw.groupby("nm_logradouro")["COD_LOGRADOURO"].nunique().max()
)

Number of nm_logradouro categories in COD_LOGRADOURO: 1
Number of COD_LOGRADOURO in nm_logradouro categories: 2


Ops, there is a `nm_logradouro` with two `COD_LOGRADOURO`. Let's check it out.

In [110]:
ser_duplicated_logradouro = df_raw.groupby("nm_logradouro")["COD_LOGRADOURO"].nunique().sort_values(ascending=False)
ser_duplicated_logradouro = ser_duplicated_logradouro[ser_duplicated_logradouro > 1]

In [111]:
df_raw[df_raw.nm_logradouro.isin(ser_duplicated_logradouro.index)].groupby("nm_logradouro")["COD_LOGRADOURO"].unique()

nm_logradouro
Avenida Albert Sabin     [213819.0, 45849.0]
Rua D                   [293167.0, 192088.0]
Rua Dez                 [138875.0, 179747.0]
Rua Gonzaguinha         [213876.0, 187690.0]
Rua Leão Veloso            [3822.0, 95109.0]
Rua Licânia               [8011.0, 136655.0]
Rua Lutécia               [61465.0, 97154.0]
Rua Maria Eugênia          [76422.0, 1768.0]
Rua Mimosa              [113795.0, 114934.0]
Rua Oito                [187054.0, 138859.0]
Rua Onze                [138883.0, 187088.0]
Rua Pedralva             [118976.0, 97295.0]
Rua Projetada A         [463828.0, 185215.0]
Rua Seis                [187039.0, 138834.0]
Rua Sete                [187047.0, 138842.0]
Name: COD_LOGRADOURO, dtype: object

This place has two 'COD_LOGRADOURO' each. Let's check if they are place with the same name but in different neighborhoods.

In [112]:
df_raw[df_raw.nm_logradouro.isin(ser_duplicated_logradouro.index)].groupby("nm_logradouro")["nm_bairro_norm"].unique()

nm_logradouro
Avenida Albert Sabin    [BARRA DA TIJUCA, RECREIO DOS BANDEIRANTES]
Rua D                                                [CAMPO GRANDE]
Rua Dez                                   [VISTA ALEGRE, PACIENCIA]
Rua Gonzaguinha                       [BARRA DA TIJUCA, PORTUGUESA]
Rua Leão Veloso                                      [TAUA, PAVUNA]
Rua Licânia                               [REALENGO, GARDENIA AZUL]
Rua Lutécia                   [IMPERIAL DE SAO CRISTOVAO, REALENGO]
Rua Maria Eugênia                           [HUMAITA, CAMPO GRANDE]
Rua Mimosa                                 [BRAZ DE PINA, CURICICA]
Rua Oito                                   [REALENGO, VISTA ALEGRE]
Rua Onze                                   [VISTA ALEGRE, REALENGO]
Rua Pedralva                                  [PACIENCIA, REALENGO]
Rua Projetada A                                      [CAMPO GRANDE]
Rua Seis                                   [REALENGO, VISTA ALEGRE]
Rua Sete                          

For most of them, they are different places in different neighborhoods, except the ones in Campo Grande.

In [123]:
df_raw[df_raw.nm_logradouro.isin(["Rua Projetada A", "Rua D"])].groupby(["ano", "nm_logradouro"]).count().COD_LOGRADOURO

ano   nm_logradouro  
2017  Rua Projetada A     4
2018  Rua Projetada A    16
2019  Rua Projetada A     9
2020  Rua D               1
      Rua Projetada A     6
2021  Rua D               1
      Rua Projetada A     7
2022  Rua D               1
      Rua Projetada A    11
2023  Rua Projetada A     9
2024  Rua Projetada A     7
2025  Rua Projetada A     5
2026  Rua Projetada A     2
Name: COD_LOGRADOURO, dtype: int64

### Property usage

Here we check whether `DESCRICAO` is the text description associated with `ID_UTILIZACAO`, and whether the relationship is one-to-one in both directions.

In [125]:
df_raw.groupby("DESCRICAO")["ID_UTILIZACAO"].nunique()

DESCRICAO
AREA COMUM         1
NAO RESIDENCIAL    1
RESIDENCIAL        1
TERRITORIAL        1
VAGA VINCULADA     1
Name: ID_UTILIZACAO, dtype: int64

In [126]:
df_raw.groupby("ID_UTILIZACAO")["DESCRICAO"].nunique()

ID_UTILIZACAO
1    1
2    1
3    1
4    1
5    1
Name: DESCRICAO, dtype: int64

### Typology
Next, we verify whether `ds_imovel_tipologia` is the text description associated with `COD_TIPOLOGIA`.

In [135]:
df_raw.groupby("ds_imovel_tipologia")["COD_TIPOLOGIA"].nunique().max()

np.int64(1)

In [136]:
df_raw.groupby("COD_TIPOLOGIA")["ds_imovel_tipologia"].nunique().max()

np.int64(1)

In [ ]:
def clean_itbi_code(value):
    if pd.isna(value):
        return value

    value = str(value)

    value = re.sub(r"^_x0031_0", "", value)

    return pd.to_numeric(value, errors="coerce")

The property-usage check shows a one-to-one relationship: each `DESCRICAO` corresponds to a single `ID_UTILIZACAO`, and each `ID_UTILIZACAO` corresponds to a single description.

## Time features
The dataset contains two versions of both year and month. 
We can compare them directly to confirm whether they carry the same information.

In [130]:
(df_raw.ano == df_raw.ano_texto).all(), (df_raw.mes == df_raw.mes_texto).all()

(np.True_, np.True_)

The values in `ano` and `mes` are consistent with `ano_texto` and `mes_texto`. These pairs therefore contain duplicated information.

The typology check above also shows a one-to-one mapping: each typology code is associated with a single description.

## Property age factor

`FATOR_IDADE` represents the age-related coefficient used in the property valuation data. 
Unlike most of the dataset, this feature contains some missing values.

Let's check whether those missing values are associated with specific property typologies.

In [179]:
df_raw[df_raw.FATOR_IDADE.isna()].groupby("ds_imovel_tipologia").count().ano

ds_imovel_tipologia
CASA            1
ESPECIAL       18
LOJA            1
TERRENO     12761
Name: ano, dtype: int64

The missing values appear to be strongly related to property typology. 
To put this into context, we can compare them with the total number of observations for the relevant typologies.

In [186]:
df_raw.groupby("ds_imovel_tipologia").COD_TIPOLOGIA.count()[["CASA", "ESPECIAL", "LOJA", "TERRENO"]]

ds_imovel_tipologia
CASA        57772
ESPECIAL     2648
LOJA         9108
TERRENO     13509
Name: COD_TIPOLOGIA, dtype: int64

Almost all observations classified as `TERRENO` have no `FATOR_IDADE`. 
This is expected because an age factor describes the age of a built structure, while vacant land does not have a construction age.

# Data treatment

The previous checks showed that several columns contain duplicated information. 
In most cases, a compact numerical or categorical code is paired with a longer text description.

For the next stages of the project, the main transaction table will keep the compact features used for analysis and machine learning. 
The text descriptions will be stored separately as lookup tables. 
This keeps the modeling dataset smaller while preserving readable labels for plots and interpretation.

## Numerical and categorical data

The DataFrame `df` contains the numerical and categorical features selected for analysis and modeling. It will be the main transaction-level dataset used in the next notebooks.

In [152]:
df = df_raw[
    [
        'ano', 'mes',        # duplicated with ano_texto and mes_texto
        'COD_REGIAO_FISCAL',
        'COD_BAIRRO',        # duplicated with nm_bairro_norm
        'COD_LOGRADOURO',    # duplicated with nm_logradouro
        'COD_TRECHO', 
        'COD_TIPOLOGIA',     # duplicated with ds_imovel_tipologia
        'ID_UTILIZACAO',     # duplicated with DESCRICAO
        'FATOR_IDADE',
        'vl_declarado', 
        'area', 
        'valor_m2'           # derived from vl_declarado and area
    ]
].copy()
df.info()

<class 'pandas.DataFrame'>
Index: 490371 entries, 0 to 79934
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ano                490371 non-null  int64  
 1   mes                490371 non-null  int64  
 2   COD_REGIAO_FISCAL  490371 non-null  str    
 3   COD_BAIRRO         490274 non-null  float64
 4   COD_LOGRADOURO     490323 non-null  float64
 5   COD_TRECHO         481702 non-null  float64
 6   COD_TIPOLOGIA      490371 non-null  float64
 7   ID_UTILIZACAO      490371 non-null  int64  
 8   FATOR_IDADE        477590 non-null  float64
 9   vl_declarado       490371 non-null  float64
 10  area               490331 non-null  float64
 11  valor_m2           490331 non-null  float64
dtypes: float64(8), int64(3), str(1)
memory usage: 49.1 MB


In [153]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ano,490371.0,2020.733871,3.216810e+00,2016.000000,2018.000000,2021.000000,2024.000000,2.026000e+03
mes,490371.0,6.702729,3.380565e+00,1.000000,4.000000,7.000000,10.000000,1.200000e+01
COD_BAIRRO,490274.0,79.957046,4.932743e+01,0.000000,28.000000,79.000000,128.000000,1.650000e+02
COD_LOGRADOURO,490323.0,96777.402455,9.170433e+04,18.000000,48579.000000,73478.000000,103549.000000,7.790400e+05
COD_TRECHO,481702.0,3.190722,6.721642e+00,0.000000,1.000000,1.000000,2.000000,1.110000e+02
COD_TIPOLOGIA,490371.0,12.010207,1.126095e+01,1.000000,10.000000,10.000000,10.000000,9.900000e+01
ID_UTILIZACAO,490371.0,1.183671,5.168694e-01,1.000000,1.000000,1.000000,1.000000,5.000000e+00
FATOR_IDADE,477590.0,0.740892,1.744941e-01,0.500000,0.550000,0.800000,0.850000,1.000000e+00
vl_declarado,490371.0,811643.370726,9.006169e+07,0.100000,220000.000000,372392.440000,690000.000000,6.302690e+10
area,490331.0,183.483669,2.638316e+03,1.000000,49.000000,70.000000,103.000000,8.775780e+05


Missing values in the `COD_` and `ID_` features are replaced with -1. 
This keeps the affected rows in the dataset while assigning missing or unknown categories a consistent identifier.

In [154]:
cod_features = ["COD_BAIRRO", "COD_LOGRADOURO", "COD_TRECHO", "COD_TIPOLOGIA", "ID_UTILIZACAO"]
df[cod_features] = df[cod_features].fillna(-1)

In [190]:
astype_dict = {
    "COD_BAIRRO" : int,
    "COD_LOGRADOURO" : int,
    "COD_TRECHO" : int,
    "COD_TIPOLOGIA" : int,
    "ID_UTILIZACAO" : int,
}
df = df.astype(astype_dict)

In [191]:
df.info()

<class 'pandas.DataFrame'>
Index: 490371 entries, 0 to 79934
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ano                490371 non-null  int64  
 1   mes                490371 non-null  int64  
 2   COD_REGIAO_FISCAL  490371 non-null  str    
 3   COD_BAIRRO         490371 non-null  int64  
 4   COD_LOGRADOURO     490371 non-null  int64  
 5   COD_TRECHO         490371 non-null  int64  
 6   COD_TIPOLOGIA      490371 non-null  int64  
 7   ID_UTILIZACAO      490371 non-null  int64  
 8   FATOR_IDADE        477590 non-null  float64
 9   vl_declarado       490371 non-null  float64
 10  area               490331 non-null  float64
 11  valor_m2           490331 non-null  float64
dtypes: float64(4), int64(7), str(1)
memory usage: 49.1 MB


## Informational data

The text labels removed from the main transaction table are kept in separate lookup DataFrames. 
These tables make it easy to translate codes back into readable neighborhood, street, typology, and usage descriptions when needed.

In [213]:
df_bairro = df_raw.groupby("COD_BAIRRO")["nm_bairro_norm"].unique()
df_bairro = df_bairro.apply(lambda x : x[0])
df_bairro = df_bairro.to_frame()
df_bairro.index = df_bairro.index.astype(int)
df_bairro

,nm_bairro_norm
COD_BAIRRO,
0,NaN
1,SAUDE
2,GAMBOA
3,SANTO CRISTO
4,CAJU
...,...
159,PARQUE COLUMBIA
161,LAPA
163,JABOUR


In [212]:
df_logradouro = df_raw.groupby("COD_LOGRADOURO")["nm_logradouro"].unique()
df_logradouro = df_logradouro.apply(lambda x : x[0])
df_logradouro = df_logradouro.to_frame()
df_logradouro.index = df_logradouro.index.astype(int)
df_logradouro

,nm_logradouro
COD_LOGRADOURO,
18,Rua Agostinho Menezes
59,Rua Professor Sá Nunes
83,Rua Francisco Furtado
91,Rua Major Souza Mendes
109,Rua Liberato Barroso
...,...
606079,Rua Mc Marcinho
606087,Rua Lolita Rodrigues
606319,VIA 1 PAA 12786 - PAL 50066


In [214]:
df_tipologia = df_raw.groupby("COD_TIPOLOGIA")["ds_imovel_tipologia"].unique()
df_tipologia = df_tipologia.apply(lambda x : x[0])
df_tipologia = df_tipologia.to_frame()
df_tipologia.index = df_tipologia.index.astype(int)
df_tipologia

,ds_imovel_tipologia
COD_TIPOLOGIA,
1,TERRENO
2,CASA
10,APARTAMENTO
19,APART-HOTEL FRENTE
28,SALA
30,APART-HOTEL FUNDOS
31,APART-HOTEL
32,UNID. HOTELEIRA
36,LOJA


In [215]:
df_descricao = df_raw.groupby("ID_UTILIZACAO")["DESCRICAO"].unique()
df_descricao = df_descricao.apply(lambda x : x[0])
df_descricao = df_descricao.to_frame()
df_descricao.index = df_descricao.index.astype(int)
df_descricao

,DESCRICAO
ID_UTILIZACAO,
1,RESIDENCIAL
2,NAO RESIDENCIAL
3,TERRITORIAL
4,VAGA VINCULADA
5,AREA COMUM


# Saving the processed data

The cleaned transaction table and the lookup tables are saved in a single Excel workbook, with one sheet for each dataset. 
This file will be used as the input for the next stages of the project.

In [216]:
savefile_path = "../data/processed/ITBI.xlsx"

with pd.ExcelWriter(savefile_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="transacoes")
    df_bairro.to_excel(writer, sheet_name="bairro")
    df_logradouro.to_excel(writer, sheet_name="logradouro")
    df_tipologia.to_excel(writer, sheet_name="tipologia")
    df_descricao.to_excel(writer, sheet_name="descricao")